<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/24_nlp_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NLP Sequence with word embeddings
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  5108k      0  0:00:16  0:00:16 --:--:-- 9466k


In [2]:
!rm -r aclImdb/train/unsup

In [3]:
import os, pathlib, shutil, random
from tensorflow import keras

# Extract data
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

# Create sets
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
# Learning word embeddings
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length = max_length,
)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [5]:
"""
Word inedx -> Embedding layer -> Corresponding word vector
embedding_layer = layers.Embedding(input_dim = max_tokens, output_dim=256)

Learning the words embeddings with the model, functions the same as
learning the weights of a neural  network. First we start with a random word
vector and then learn the place in the vector space.
"""

inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
callbacks = [
  keras.callbacks.ModelCheckpoint("embeddings_bidir_gru.keras",
  save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
          callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 41ms/step - accuracy: 0.8270 - loss: 0.4261 - val_accuracy: 0.8600 - val_loss: 0.3457
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.8743 - loss: 0.3288 - val_accuracy: 0.7732 - val_loss: 0.5854
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 44s 46ms/step - accuracy: 0.8966 - loss: 0.2800 - val_accuracy: 0.8584 - val_loss: 0.3720
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 42ms/step - accuracy: 0.9133 - loss: 0.2437 - val_accuracy: 0.8648 - val_loss: 0.4007
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.9309 - loss: 0.1970 - val_accuracy: 0.8796 - val_loss: 0.3285
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.9416 - loss: 0.1705 - val_accuracy: 0.8660 - val_loss: 0.3843
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.9552 - loss: 0.1394 - val_accuracy: 0.8788 - val_loss: 0.3647
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9582 - loss: 0.1328 - 

In [9]:
model = keras.models.load_model("embeddings_bidir_gru.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8642 - loss: 0.3459
Test acc: 0.864


- Problem:
  - Our input sequences are full of zeros!
  - The TextVectorization layers truncates text that is longer than 600, and the shorter sentences get padded with ZEROS.

  - The Bidirectional Layers run in parrallel form the back and from the beginning:
    - In practice the layers not just waste their time, but the information in the internal state of the RNN will fade out!
    -  Thats what masking is for!

- Simple example:
- [[4, 3, 2, 1, 0, 0, 0],
- [5, 4, 3, 2, 1, 0, 0],
- [2, 1, 0, 0, 0, 0, 0]]

- corresponding **Mask**
- [[ True, True, True, True, False, False, False],
- [ True, True, True, True, True, False, False],
- [ True, True, False, False, False, False, False]]



In [10]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(
    input_dim=max_tokens,
    output_dim=256,
    mask_zero=True
    )(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  5,120,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     73,984 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
callbacks = [
  keras.callbacks.ModelCheckpoint("embeddings_bidir_gru_with_masking.keras",
  save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
          callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 45ms/step - accuracy: 0.6924 - loss: 0.5654 - val_accuracy: 0.8464 - val_loss: 0.3601
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.8666 - loss: 0.3223 - val_accuracy: 0.8770 - val_loss: 0.3094
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.8980 - loss: 0.2545 - val_accuracy: 0.8742 - val_loss: 0.3264
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 43s 46ms/step - accuracy: 0.9261 - loss: 0.1977 - val_accuracy: 0.8726 - val_loss: 0.3629
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 39s 43ms/step - accuracy: 0.9432 - loss: 0.1494 - val_accuracy: 0.8686 - val_loss: 0.3726
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 44ms/step - accuracy: 0.9606 - loss: 0.1100 - val_accuracy: 0.8764 - val_loss: 0.4829
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 43ms/step - accuracy: 0.9709 - loss: 0.0842 - val_accuracy: 0.8762 - val_loss: 0.4391
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 43ms/step - accuracy: 0.9783 - loss: 0.0592 - 

In [12]:
model = keras.models.load_model("embeddings_bidir_gru_with_masking.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.8789 - loss: 0.2997
Test acc: 0.876
